Trains the user model (data/model_checkpoints/netmedgpt.pt) on all edges of the curated KG (nodes_preprocessed.csv,
edges_preprocessed.csv): no train / validation / test split. Adapted from BioMedFormer
04-train_biomedformer_transductive_with_all_data_after_filtering_falsedrugadrlinks.ipynb to the NetMedGPT folder.
Run from the NetMedGPT folder (netmedgpt environment).

In [1]:
inference = "transductive"
quiried_edge_types = ['indication', 'contraindication', 'off-label use', 'drug_protein']
quiried_node_types = ['disease|drug', 'disease|drug', 'disease|drug', 'drug|gene/protein']

Epoch = 95
Repeat = 1

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import os
import subprocess
import json
from tqdm import tqdm
import time
import itertools
from datetime import datetime

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
import matplotlib.pyplot as plt

from torch.optim.lr_scheduler import LambdaLR
import math

from netmedgpt.model import TransformerModel, create_mask
from netmedgpt.sentence_generation_all_data import sentence_generation

In [4]:
print(os.environ.get('CONDA_DEFAULT_ENV'))
nvidia_smi_output = subprocess.check_output(['nvidia-smi']).decode('utf-8')
print(nvidia_smi_output)
torch.cuda.is_available()

netmedgpt
Wed Nov 18 18:19:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.35.05              Driver Version: 560.35.05      CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 PCIe               Off |   00000000:17:00.0 Off |                    0 |
| N/A   43C    P0            195W /  350W |   80686MiB /  81559MiB |     85%      Default |
|                                         |                        |             Disabled |
+-------------------------------------

True

In [ ]:
GPU = 0
device = torch.device(f'cuda:{GPU}' if torch.cuda.is_available() else 'cpu')

with open("data/parameters.json", 'r') as file:
    all_param = json.load(file)

data_dir = all_param['files']['data_dir']
KG_dir = os.path.join(data_dir, "KG")

os.makedirs(os.path.join(data_dir, "log"), exist_ok=True)
log_dir = os.path.join(data_dir, "log")

os.makedirs(os.path.join(data_dir, "model_checkpoints"), exist_ok=True)
model_dir = os.path.join(data_dir, "model_checkpoints")

In [ ]:
# curated KG (identical to nodes_filtered.csv / edges_filtered_false_drug_adr_removed.csv of BioMedFormer)
nodes = pd.read_csv(os.path.join(KG_dir, 'nodes_preprocessed.csv'), sep= ',')
edge = pd.read_csv(os.path.join(KG_dir, "edges_preprocessed.csv"))

In [ ]:
# hyperparameters
test_ratio = 0.00
val_ratio = 0.00
seed_base = 42

In [ ]:
# dictionary of relations and their node sources
relation_node_types = {
    'protein_protein': ('NCBI', 'NCBI'),
    'drug_protein': ('DrugBank', 'NCBI'),
    'contraindication':('MONDO', 'DrugBank'),
    'indication': ('MONDO', 'DrugBank'),
    'off-label use': ('MONDO', 'DrugBank'),
    'drug_drug': ('DrugBank', 'DrugBank'),
    'phenotype_protein': ('HPO', 'NCBI'),
    'phenotype_phenotype': ('HPO', 'HPO'),
    'disease_phenotype_negative': ('MONDO', 'HPO'),
    'disease_phenotype_positive': ('MONDO', 'HPO'),
    'disease_protein': ('MONDO', 'NCBI'),
    'disease_disease': ('MONDO', 'MONDO'),
    'drug_effect': ('DrugBank', 'HPO'),
    'bioprocess_bioprocess': ('GO', 'GO'),
    'molfunc_molfunc': ('GO', 'GO'),
    'cellcomp_cellcomp': ('GO', 'GO'),
    'molfunc_protein': ('GO', 'NCBI'),
    'cellcomp_protein': ('GO', 'NCBI'),
    'bioprocess_protein': ('GO', 'NCBI'),
    'exposure_protein': ('CTD', 'NCBI'),
    'exposure_disease': ('CTD', 'MONDO'),
    'exposure_exposure': ('CTD', 'CTD'),
    'exposure_bioprocess': ('CTD', 'GO'),
    'exposure_molfunc': ('CTD', 'GO'),
    'exposure_cellcomp': ('CTD', 'GO'),
    'pathway_pathway': ('REACTOME', 'REACTOME'),
    'pathway_protein': ('REACTOME', 'NCBI'),
    'anatomy_anatomy': ('UBERON', 'UBERON'),
    'anatomy_protein_present': ('UBERON', 'NCBI'),
    'anatomy_protein_absent': ('UBERON', 'NCBI'),
}

In [ ]:
all_ids = {
    'drug': torch.tensor(
        nodes.loc[nodes['node_type'] == 'drug', 'node_index'].unique(), dtype=torch.int64
    ),
    'gene': torch.tensor(
        nodes.loc[nodes['node_type'] == 'gene/protein', 'node_index'].unique(), dtype=torch.int64
    ),
    'phenotype': torch.tensor(
        nodes.loc[nodes['node_type'] == 'effect/phenotype', 'node_index'].unique(), dtype=torch.int64
    )
}

# Get mapping from node_types to z_index values
node_type_to_z = edge.groupby('node_types')['z_index'].unique().to_dict()
all_ids_node2 = {}

for z in node_type_to_z.get('disease|drug', []):
    all_ids_node2[z] = all_ids['drug']

for z in node_type_to_z.get('drug|gene/protein', []):
    all_ids_node2[z] = all_ids['gene']

for z in node_type_to_z.get('drug|effect/phenotype', []):
    all_ids_node2[z] = all_ids['phenotype']


In [ ]:
# other hyperparameters
mask_token = edge['z_index'].max() +1
vocab_size = mask_token + 1

relation_type = list(edge['relation'].unique())
node_types = list(nodes['node_type'].unique())
node_types_without_feat = ['biological_process',
                           'molecular_function',
                           'cellular_component',
                           'exposure',
                           'pathway',
                           'anatomy']

mask = ['mask']
entity = node_types + relation_type + mask

without_feat = {
    'node': node_types_without_feat,
    'relation': relation_type,
    'mask': mask
}

## find relation_mask_index
relation_index = edge.loc[edge['relation'].isin(relation_type), ['relation', 'z_index']].drop_duplicates()
mask_row = pd.DataFrame([['mask', mask_token]], columns=['relation', 'z_index'])
relation_mask_index = pd.concat([relation_index, mask_row], ignore_index=True)

In [ ]:
# hyperparameters (best ray-tune configuration, stored in data/parameters.json)
best_hyperparam = dict(all_param['best_hyperparam'])
print(best_hyperparam)

In [ ]:
# node features of the curated KG (PubMedBERT, fingerprints, ESM2 and random features of the node types without
# attributes), precomputed in data/KG/embeddings_preprocessedKG.pt
feat = torch.load(os.path.join(KG_dir, "embeddings_preprocessedKG.pt"))

In [ ]:
# relations and the mask token get fixed (non-trainable) vectors of ones, as in the training of netmedgpt.pt
for k in without_feat['relation'] + without_feat['mask']:
    feat[k] = {'fixed': torch.ones(1, best_hyperparam['hidden_channels'])}

In [ ]:
def lr_lambda(current_step):
    if current_step < warmup_steps:
        return float(current_step) / float(max(1, warmup_steps))
    progress = float(current_step - warmup_steps) / float(max(1, total_steps - warmup_steps))
    return 0.5 * (1.0 + math.cos(math.pi * progress))

In [ ]:
# Loop over Repeat
for i in tqdm(range(Repeat)):
    seed = seed_base + i
    print(f'\nRepeat: {i}, Seed: {seed}')

    data = sentence_generation(edge, nodes, quiried_node_types, quiried_edge_types, test_ratio, val_ratio, relation_node_types, all_param, seed)

    # walks and validation and test data
    node2vec_walks = data['node2vec_walks']
    log_name = f"netmedgpt_user_model_{inference}_wl{all_param['node2vec']['walk_length']}_wpn{all_param['node2vec']['walks_per_node']}_dim{best_hyperparam['hidden_channels']}_head{best_hyperparam['nhead']}_lencoder{best_hyperparam['N_encoder_layers']}_bs{best_hyperparam['batch_size']}_lr{best_hyperparam['learning_rate']}_seed{seed}"
    # log_name = f"netmedgpt_atr_final_filteredKG_drugdrug_drugadr_drugwithmanyindications_removed_{inference}_wl{all_param['node2vec']['walk_length']}_wpn{all_param['node2vec']['walks_per_node']}_dim{best_hyperparam['hidden_channels']}_head{best_hyperparam['nhead']}_lencoder{best_hyperparam['N_encoder_layers']}_bs{best_hyperparam['batch_size']}_lr{best_hyperparam['learning_rate']}_seed{seed}"

    log_file_prob = f"{log_dir}/{log_name}.log"
    model_save_path = f"{model_dir}/{log_name}.pt"
    seq_len = (all_param['node2vec']['walk_length']*2)-1   # walk_length_with_relation

    dataset = TensorDataset(node2vec_walks)
    dataloader = DataLoader(dataset, batch_size=best_hyperparam['batch_size'], shuffle=True)

    model = TransformerModel(
        vocab_size,
        best_hyperparam['hidden_channels'],
        best_hyperparam['nhead'],
        best_hyperparam['N_encoder_layers'],
        (all_param['node2vec']['walk_length']*2)-1, #walk_length_with_relation
        device=device,
        feat = feat,
        nodes = nodes,
        entity = entity,
        relation_mask_index = relation_mask_index,
        pos_emb='fixed',
    ).to(device)

    optimizer = optim.Adam(model.parameters(), lr=best_hyperparam['learning_rate'])
    scaler = GradScaler()

    warmup_steps = int(0.1 * Epoch * len(dataloader))  # 10% warmup
    total_steps = Epoch * len(dataloader)
    
    scheduler = LambdaLR(optimizer, lr_lambda)    
    criterion = nn.CrossEntropyLoss(ignore_index=vocab_size)

    for epoch in range(Epoch):
        model.train()
        epoch_loss = 0

        for batch in dataloader:
            optimizer.zero_grad()
            input_batch = batch[0].to(device)
            masked_input, mask = create_mask(input_batch, vocab_size, mask_token_id=mask_token)
        
            with autocast():  # AMP START
                output = model(masked_input)
                loss = criterion(output[mask], input_batch[mask])
                
            # AMP END
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            batch_loss = loss.item()
            epoch_loss += batch_loss

        avg_loss = epoch_loss / len(dataloader)

        # Logging
        with open(log_file_prob, "a") as f:
            f.write(f"Epoch {epoch+1}:\n")
            f.write(f"Loss_train: {avg_loss}\n")


In [ ]:
best_hyperparam['walk_length'] = 5
best_hyperparam['walk_per_nodes'] = 30

In [ ]:
# saved next to netmedgpt.pt under a new name (netmedgpt.pt is not overwritten)
checkpoint_path = f"{model_dir}/{log_name}_modelLastEpoch.pt"

# Save the model
checkpoint_dict = {
    'epoch': 95,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'loss': loss,
    'parameters' : best_hyperparam
}

torch.save(checkpoint_dict, checkpoint_path)
print(f"Checkpoint saved at {checkpoint_path}")

In [ ]:
# import matplotlib.pyplot as plt
# import seaborn as sns

# sns.boxplot(data=LP_result_val_df_all, x='seed', y='auprc')
# plt.title('AUPRC across seeds (val)')
# plt.savefig(os.path.join(config_dir, 'boxplot_val_auprc.png'))
